**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/sumahegde.work@outlook.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

bronze silver gold


**Access-key**

In [0]:
dbutils.widgets.text("catalog", "databricksmaster", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/{data_source}/"

print(base_path)

abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/


## Bronze

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("recursiveFileLookup", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .withColumn("file_name", F.expr("_metadata.file_path"))
)

display(df)

customer_id,customer_name,city,read_timestamp,file_name
789201,FitFuel Market,Bengaluru,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789202,FitFuel Market,Hyderabad,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789203,FitFuel Market,New Delhi,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789301,Athlete's Choice Store,Bengaluru,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789303,Athlete's Choice Store,New Delhi,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789101,Endurance Foods,Bengalore,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789102,Endurance Foods,Hyderabad,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789103,Endurance Foods,New Delhi,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789121,HydroBoost Nutrition,Hyderabad,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789122,HydroBoost Nutrition,New Delhi,2026-05-07T06:15:05.903Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv


In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)



In [0]:
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name
789201,FitFuel Market,Bengaluru,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789202,FitFuel Market,Hyderabad,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789203,FitFuel Market,New Delhi,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789301,Athlete's Choice Store,Bengaluru,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789303,Athlete's Choice Store,New Delhi,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789101,Endurance Foods,Bengalore,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789102,Endurance Foods,Hyderabad,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789103,Endurance Foods,New Delhi,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789121,HydroBoost Nutrition,Hyderabad,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789122,HydroBoost Nutrition,New Delhi,2026-05-07T06:18:13.733Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv


In [0]:
df.write \
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+--------------------+
|customer_id|       customer_name|     city|      read_timestamp|           file_name|
+-----------+--------------------+---------+--------------------+--------------------+
|     789201|      FitFuel Market|Bengaluru|2026-05-07 06:22:...|abfss://source@my...|
|     789202|      FitFuel Market|Hyderabad|2026-05-07 06:22:...|abfss://source@my...|
|     789203|      FitFuel Market|New Delhi|2026-05-07 06:22:...|abfss://source@my...|
|     789301|Athlete's Choice ...|Bengaluru|2026-05-07 06:22:...|abfss://source@my...|
|     789303|Athlete's Choice ...|New Delhi|2026-05-07 06:22:...|abfss://source@my...|
|     789101|     Endurance Foods|Bengalore|2026-05-07 06:22:...|abfss://source@my...|
|     789102|     Endurance Foods|Hyderabad|2026-05-07 06:22:...|abfss://source@my...|
|     789103|     Endurance Foods|New Delhi|2026-05-07 06:22:...|abfss://source@my...|
|     789121| HydroBoost Nutri...|Hyderabad

**Transformations**

- 1: Drop Duplicates

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

customer_id,count
789522,2
789603,2
789321,2
789503,2


In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

Rows before duplicates dropped:  39
Rows after duplicates dropped:  35


- 2: Trim spaces in customer name

In [0]:
# check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name
789121,HydroBoost Nutrition,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789401,SprintX nutrition,Bengaluru,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789420,ZenAthlete foods,null,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789421,ZenAthlete Foods,Hyderbad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789521,PrimeFuel Nutrition,null,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789702,StaminaX Store,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv


In [0]:
## remove those trim values

df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
# # Sanity Check

# # check those values
# display(
#     df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
# )


customer_id,customer_name,city,read_timestamp,file_name


- 3: Data Quality Fix: Correcting City Typos

In [0]:
df_silver.select('city').distinct().show()

+----------+
|      city|
+----------+
|  Hyderbad|
|      NULL|
|Bengaluruu|
| NewDelhee|
|Hyderabadd|
|  NewDheli|
| Bengalore|
| Bengaluru|
| New Delhi|
| Hyderabad|
|  NewDelhi|
+----------+



In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
# Sanity check
df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|     NULL|
|Bengaluru|
|New Delhi|
|Hyderabad|
+---------+



- 4: Fix Title-Casing Issue

In [0]:
df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|   SprintX nutrition|
|    ZenAthlete Foods|
|   Champion's choice|
|       Recovery Lane|
|EliteAthlete Nutr...|
|      StaminaX Store|
|MacroBite superfoods|
| PrimeFuel Nutrition|
|Peak performance ...|
|Peak Performance ...|
|      PowerSnack hub|
|   SprintX Nutrition|
|     Endurance Foods|
|    ZenAthlete foods|
|   champion's Choice|
|      GamePlan Foods|
|Athlete's Choice ...|
|      FitFuel Market|
|MacroBite Superfoods|
|HydroBoost Nutrition|
+--------------------+
only showing top 20 rows



In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

In [0]:
# sanity check

df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|       Recovery Lane|
|Eliteathlete Nutr...|
|Macrobite Superfoods|
|Peak Performance ...|
|      Gameplan Foods|
| Primefuel Nutrition|
|     Endurance Foods|
|Hydroboost Nutrition|
|   Sprintx Nutrition|
|Athlete's Choice ...|
|      Fitfuel Market|
|    Zenathlete Foods|
|   Champion's Choice|
|      Powersnack Hub|
|      Staminax Store|
+--------------------+



- 5: Handling missing cities

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

+-----------+-------------------+----+-----------------------+------------------------------------------------------------------------------------------+
|customer_id|customer_name      |city|read_timestamp         |file_name                                                                                 |
+-----------+-------------------+----+-----------------------+------------------------------------------------------------------------------------------+
|789403     |Sprintx Nutrition  |NULL|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789420     |Zenathlete Foods   |NULL|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789521     |Primefuel Nutrition|NULL|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789603     |Recovery Lane      |NULL|2026-05-07 06:22:12.631|abfss://source

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+-----------------------+------------------------------------------------------------------------------------------+
|customer_id|customer_name      |city     |read_timestamp         |file_name                                                                                 |
+-----------+-------------------+---------+-----------------------+------------------------------------------------------------------------------------------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789402     |Sprintx Nutrition  |Hyderabad|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789403     |Sprintx Nutrition  |NULL     |2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789420     |Zenathlete Foods   |NULL     |202

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

display(df_silver)

customer_id,customer_name,city,read_timestamp,file_name
789101,Endurance Foods,Bengaluru,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789102,Endurance Foods,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789103,Endurance Foods,New Delhi,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789121,Hydroboost Nutrition,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789122,Hydroboost Nutrition,New Delhi,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789201,Fitfuel Market,Bengaluru,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789202,Fitfuel Market,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789203,Fitfuel Market,New Delhi,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789220,Macrobite Superfoods,Bengaluru,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv
789221,Macrobite Superfoods,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv


In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+-----------------------+------------------------------------------------------------------------------------------+
|customer_id|customer_name      |city     |read_timestamp         |file_name                                                                                 |
+-----------+-------------------+---------+-----------------------+------------------------------------------------------------------------------------------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789402     |Sprintx Nutrition  |Hyderabad|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789403     |Sprintx Nutrition  |New Delhi|2026-05-07 06:22:12.631|abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv|
|789420     |Zenathlete Foods   |Bengaluru|202

- 6: Convert customer_id to string

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)

None


### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,customer,market,platform,channel
789101,Endurance Foods,Bengaluru,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv,Endurance Foods-Bengaluru,India,Sports Bar,Acquisition
789102,Endurance Foods,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods,New Delhi,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789121,Hydroboost Nutrition,Hyderabad,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv,Hydroboost Nutrition-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,2026-05-07T06:22:12.631Z,abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/customers/customers.csv,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merging Data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, "databricksmaster.gold.dim_customers")
df_child_customers = spark.table("databricksmaster.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from gold.dim_customers;

customer_code,customer,market,platform,channel
789101,Endurance Foods-Bengaluru,India,Sports Bar,Acquisition
789102,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789121,Hydroboost Nutrition-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789203,Fitfuel Market-New Delhi,India,Sports Bar,Acquisition
789220,Macrobite Superfoods-Bengaluru,India,Sports Bar,Acquisition
789221,Macrobite Superfoods-Hyderabad,India,Sports Bar,Acquisition
